In [ ]:
#Construct a trading strategy using NN, foe 1 stock and the  the whole universe
#Use our Protfolio Simulator

In [ ]:
import sys, os
while not os.path.isdir('src') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')
sys.path.insert(0, 'src')
 
import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path
 
from tradinglab.data_feed import DataFeed
from tradinglab.features import feature_columns, build_pooled_dataset, N_FEATURES
from tradinglab.models import MLP
from tradinglab.ml import train_model, predict
from tradinglab.simulator import PortfolioSimulator
from tradinglab.backtester import run_backtest

In [ ]:
# ---- 1. Discover the universe and load it ----
all_csvs = sorted(p.stem for p in Path('data/egx').glob('*.csv'))
print('files found:', all_csvs)
 
# egx30.csv is the INDEX, not a tradeable stock -- exclude it from the universe.
stock_symbols = [s for s in all_csvs if s.lower() != 'egx30']
 
feed = DataFeed.from_dir('data/egx', symbols=stock_symbols)
print('universe loaded:', feed.symbols)
print('date range:', feed.dates[0].date(), '->', feed.dates[-1].date(), f'({feed.n_days} days)')

In [ ]:
# ---- 2. Pick one stock, and one split point for everything ----
# Both models train ONLY on days before split_day, and both backtests start
# exactly at split_day -- so a model is never evaluated on a day it was
# trained on. 70% of the calendar, same convention as every other notebook.
ticker = 'ABUK'                       # change to 'HRHO' or any symbol printed above
asset_idx = feed.symbols.index(ticker)
 
split_day = int(feed.n_days * 0.7)
print(f'split day: {split_day}  ({feed.dates[split_day].date()})')
print(f'train:  {feed.dates[0].date()} -> {feed.dates[split_day - 1].date()}')
print(f'test:   {feed.dates[split_day].date()} -> {feed.dates[-1].date()}')

In [ ]:
# ---- 3. Strategy 1 -- single stock ----
# Train an MLP on ABUK alone, using the SAME day-based train/test masking
# build_pooled_dataset uses internally -- the calendar-cutoff equivalent of
# build_dataset + train_test_split, aligned exactly to split_day so it
# matches the backtest's start point precisely.
def build_single_asset_by_day(feed, asset, split_day):
    X_full = feature_columns(feed, asset)
    y_full = np.full(feed.n_days, np.nan)
    y_full[:-1] = feed.returns[1:, asset]
    days = np.arange(feed.n_days)
    valid = ~np.isnan(X_full).any(axis=1) & ~np.isnan(y_full)
    train_mask = valid & (days < split_day)
    test_mask = valid & (days >= split_day)
    return (X_full[train_mask].astype(np.float32), y_full[train_mask].astype(np.float32),
            X_full[test_mask].astype(np.float32), y_full[test_mask].astype(np.float32))
 
Xtr_single, ytr_single, Xte_single, yte_single = build_single_asset_by_day(feed, asset_idx, split_day)
print(f'{ticker} single-stock model: train {len(Xtr_single)}   test {len(Xte_single)}')
 
torch.manual_seed(0)
single_model = MLP(n_features=N_FEATURES, hidden=32)
single_history = train_model(single_model, Xtr_single, ytr_single, Xte_single, yte_single,
                              epochs=300, lr=0.01)
print(f'final train loss: {single_history["train"][-1]:.6f}   final test loss: {single_history["test"][-1]:.6f}')

In [ ]:
# ---- 4. Strategy 1 -- turn predictions into weights ----
def single_stock_strategy(obs):
    # obs: (n_assets, lookback, N_FEATURES). Only the latest timestep for our
    # one target asset is used -- the model was trained on point-in-time
    # features, not a window.
    feats = obs[asset_idx, -1, :].astype(np.float32).reshape(1, -1)
    pred = predict(single_model, feats)[0]
    weights = np.zeros(feed.n_assets)
    if pred > 0:
        weights[asset_idx] = 1.0
    else:
        weights[:] = 1.0 / feed.n_assets   # no-signal fallback -- see note above
    return weights

In [ ]:
# ---- 5. Strategy 2 -- whole universe ----
# One MLP, trained POOLED across every stock's rows before split_day
# (build_pooled_dataset already handles the per-stock day-masking so no
# stock's future rows leak into training). At inference, the same model
# scores every asset independently each day; go long the top 5 stocks with a
# positive predicted return, equal-weighted among them.
Xtr_pool, ytr_pool, Xte_pool, yte_pool = build_pooled_dataset(feed, split_day)
print(f'pooled model: train {len(Xtr_pool)}   test {len(Xte_pool)}')
 
torch.manual_seed(0)
pooled_model = MLP(n_features=N_FEATURES, hidden=32)
pooled_history = train_model(pooled_model, Xtr_pool, ytr_pool, Xte_pool, yte_pool,
                              epochs=300, lr=0.01)
print(f'final train loss: {pooled_history["train"][-1]:.6f}   final test loss: {pooled_history["test"][-1]:.6f}')

In [ ]:
# ---- 6. Strategy 2 -- turn predictions into weights ----
TOP_N = 5
 
def universe_strategy(obs):
    # obs: (n_assets, lookback, N_FEATURES). Score every asset's latest
    # features with the same pooled model, then go long the best TOP_N with
    # a positive predicted return.
    feats = obs[:, -1, :].astype(np.float32)             # (n_assets, N_FEATURES)
    preds = predict(pooled_model, feats)                  # (n_assets,)
    weights = np.zeros(feed.n_assets)
    ranked = np.argsort(preds)[::-1][:TOP_N]
    positive = ranked[preds[ranked] > 0]
    if len(positive) == 0:
        weights[:] = 1.0 / feed.n_assets                  # no-signal fallback
    else:
        weights[positive] = 1.0 / len(positive)
    return weights

In [ ]:
# ---- 7. Run both through the Portfolio Simulator ----
# lookback=1 -- both strategies only ever look at the latest timestep of the
# observation. Backtest starts exactly at split_day, so this is a genuinely
# out-of-sample evaluation for both models.
sim = PortfolioSimulator(feed, benchmark='equal_weight')
 
result_single = run_backtest(sim, single_stock_strategy, lookback=1, start=split_day)
result_universe = run_backtest(sim, universe_strategy, lookback=1, start=split_day)

In [ ]:
# ---- 8. Plots: strategy vs benchmark, for both strategies ----
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
 
axes[0].plot(result_single['dates'], result_single['portfolio'], label=f'{ticker} single-stock strategy')
axes[0].plot(result_single['dates'], result_single['benchmark'], label='equal-weight benchmark', linestyle='--')
axes[0].legend(); axes[0].grid(alpha=.3); axes[0].set_title('Single-stock strategy vs benchmark')
axes[0].tick_params(axis='x', rotation=30)
 
axes[1].plot(result_universe['dates'], result_universe['portfolio'], label='whole-universe strategy')
axes[1].plot(result_universe['dates'], result_universe['benchmark'], label='equal-weight benchmark', linestyle='--')
axes[1].legend(); axes[1].grid(alpha=.3); axes[1].set_title('Whole-universe strategy vs benchmark')
axes[1].tick_params(axis='x', rotation=30)
 
plt.tight_layout(); plt.show()

In [ ]:
# ---- 9. Did either strategy actually beat the benchmark? ----
# Total return alone can be misleading (a strategy can get lucky). Annualized
# volatility and a simple Sharpe ratio (assuming ~252 trading days, 0%
# risk-free rate) give a fuller picture of whether the return came with more
# or less risk than just holding the market.
def summarize(name, returns, value_curve):
    total_return = value_curve[-1] - 1.0
    ann_vol = float(np.std(returns) * np.sqrt(252))
    ann_return = float(np.mean(returns) * 252)
    sharpe = ann_return / ann_vol if ann_vol > 0 else float('nan')
    print(f"{name:28s}  total return {total_return:+8.2%}   ann. vol {ann_vol:7.2%}   Sharpe {sharpe:6.2f}")
 
summarize(f'{ticker} single-stock strategy', result_single['portfolio_returns'], result_single['portfolio'])
summarize('  benchmark (same period)', result_single['benchmark_returns'], result_single['benchmark'])
print()
summarize('whole-universe strategy', result_universe['portfolio_returns'], result_universe['portfolio'])
summarize('  benchmark (same period)', result_universe['benchmark_returns'], result_universe['benchmark'])